# Clear_Vision — VQ-VAE Restoration (local)

**Target:** PSNR >= 31.59 | SSIM >= 0.8415 | LPIPS <= 0.113

**Runtime:** Local **T4 GPU** via VS Code Colab extension (no Google Drive, no cloning).

**Data:** Existing `Data/Clean` and `Data/Corrupted` only — nothing scraped or re-generated.

**Strategy:** base=32, EMA VQ K=256, L1+L2+perceptual+VQ loss, AdamW+cosine LR, AMP on GPU, checkpoints in `./checkpoints/`.


## 0 — Local paths & GPU

**`ROOT`** = the folder that contains **both** `Data/` and `ClearVision/` (your VS Code project folder, e.g. `CLEAR_VISION` or `Clear_Vision`).

```
ROOT/                    ← PROJECT_ROOT
├── Data/
│   ├── Clean/           ← training targets
│   └── Corrupted/       ← model inputs
├── ClearVision/         ← Python code (VQ-VAE, etc.)
├── final.ipynb
└── checkpoints/         ← created when training runs
```

- **Colab cloud:** upload the project, then in cell 0 set `CD_TO = "/content/CLEAR_VISION"` (or your upload path).
- **Local Python kernel:** set `CD_TO = r"d:\IIMA_Show\ClearVision\Clear_Vision"`.
- Leave `CD_TO = None` to auto-search.


In [1]:
import os
import sys
from pathlib import Path

# ── 1) cd into the project folder (edit ONE line for your setup) ─────────────
# Colab cloud — after uploading the whole project folder to /content:
CD_TO = "D:\IIMA_Show\ClearVision\content\Clear_Vision"
# Local Windows + local Python kernel (not Colab cloud):
# CD_TO = r"d:\IIMA_Show\ClearVision\Clear_Vision"
# Auto-search (leave as None):
# CD_TO = None


def _has_data_folders(root: Path) -> bool:
    return (root / "Data" / "Clean").is_dir() and (root / "Data" / "Corrupted").is_dir()


def chdir_to_project() -> Path:
    """Change working directory to the folder that contains Data/ and ClearVision/."""
    if CD_TO is not None:
        raw = str(CD_TO)
        if os.name != "nt" and len(raw) >= 2 and raw[1] == ":":
            raise FileNotFoundError(
                "CD_TO is a Windows path (d:\\...) but this kernel is Linux (Colab).\n"
                "Upload the project to /content/CLEAR_VISION and set:\n"
                "  CD_TO = '/content/CLEAR_VISION'"
            )
        target = Path(CD_TO).expanduser()
        if not target.is_absolute():
            target = (Path.cwd() / target).resolve()
        else:
            target = target.resolve()
        if not _has_data_folders(target):
            raise FileNotFoundError(
                f"After cd, expected Data/Clean under:\n  {target}\n"
                "Upload the full CLEAR_VISION folder or fix CD_TO."
            )
        os.chdir(target)
        return target

    # Auto: try common locations
    candidates = []
    cwd = Path.cwd()
    for name in ("CLEAR_VISION", "Clear_Vision"):
        candidates.extend([cwd, cwd / name])
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.is_dir():
            for name in ("CLEAR_VISION", "Clear_Vision"):
                candidates.append(base / name)
    if os.name == "nt":
        candidates.append(Path(r"d:\IIMA_Show\ClearVision\Clear_Vision"))

    tried = []
    for c in candidates:
        try:
            c = c.resolve()
        except OSError:
            continue
        tried.append(c)
        if _has_data_folders(c):
            os.chdir(c)
            return c

    msg = "\n  ".join(str(p) for p in tried[:12])
    raise FileNotFoundError(
        "Could not cd to project root.\n"
        f"  cwd was: {cwd}\n"
        "  Set CD_TO explicitly, e.g. CD_TO = '/content/CLEAR_VISION'\n"
        f"  Tried:\n  {msg}"
    )


ROOT = chdir_to_project()
print(f"cd OK  →  {ROOT}")
print(f"cwd    →  {Path.cwd()}")

CLEARVISION_PKG = ROOT / "ClearVision"
for p in (ROOT, CLEARVISION_PKG):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

CLEAN_DIR = ROOT / "Data" / "Clean"
CORRUPT_DIR = ROOT / "Data" / "Corrupted"
CKPT_DIR = ROOT / "checkpoints"
LOG_DIR = ROOT / "logs"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert CLEAN_DIR.is_dir(), f"Missing: {CLEAN_DIR}"
assert CORRUPT_DIR.is_dir(), f"Missing: {CORRUPT_DIR}"

_ext = {".jpg", ".jpeg", ".png", ".webp"}
_clean = {f for f in os.listdir(CLEAN_DIR) if Path(f).suffix.lower() in _ext}
_corrupt = {f for f in os.listdir(CORRUPT_DIR) if Path(f).suffix.lower() in _ext}
n_pairs = len(_clean & _corrupt)
assert n_pairs > 0, "No matching filenames between Clean and Corrupted"

print(f"ROOT        : {ROOT}")
print(f"Paired imgs : {n_pairs}")
print(f"Checkpoints : {CKPT_DIR}")
print(f"Logs        : {LOG_DIR}")


cd OK  →  D:\IIMA_Show\ClearVision\content\Clear_Vision
cwd    →  D:\IIMA_Show\ClearVision\content\Clear_Vision
ROOT        : D:\IIMA_Show\ClearVision\content\Clear_Vision
Paired imgs : 5335
Checkpoints : D:\IIMA_Show\ClearVision\content\Clear_Vision\checkpoints
Logs        : D:\IIMA_Show\ClearVision\content\Clear_Vision\logs


In [3]:
import os
from pathlib import Path

print("os.name:", os.name)
print("cwd:", Path.cwd())

# Check if /content exists and what's in it
for p in [Path("/content"), Path("/mnt"), Path("/home")]:
    if p.exists():
        print(f"\n{p}/")
        for item in p.iterdir():
            print(f"  {item}")

os.name: nt
cwd: D:\IIMA_Show\ClearVision\content\Clear_Vision


In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No CUDA — select T4 in the Colab kernel picker, then re-run.")


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## 1 — Dependencies (run once if needed)


In [ ]:
# %pip install torch torchvision lpips tqdm Pillow scikit-image matplotlib


## 2 — Hyperparameters


In [ ]:
CFG = dict(
    # ── Model ────────────────────────────────────────────────────
    base            = 32,      # bottleneck = base*16 = 512ch  (T4-safe)
    num_embeddings  = 256,     # codebook size K
    # embedding_dim auto = base*16 = 512
    commitment_beta = 0.25,    # β — commitment loss weight
    ema_decay       = 0.99,    # γ — EMA codebook update decay

    # ── Loss weights ─────────────────────────────────────────────
    lambda_l1       = 1.0,
    lambda_l2       = 0.1,
    lambda_perc     = 0.1,     # VGG perceptual; set 0.0 to disable

    # ── Training ─────────────────────────────────────────────────
    epochs          = 100,
    batch_size      = 16,      # safe for T4 + AMP + base=32
    lr              = 2e-4,
    weight_decay    = 1e-5,
    val_fraction    = 0.1,
    num_workers     = 0,       # 0 on Windows; try 2 on Linux
    seed            = 42,

    # ── Paths ────────────────────────────────────────────────────
    clean_dir       = CLEAN_DIR,
    corrupt_dir     = CORRUPT_DIR,
    ckpt_dir        = CKPT_DIR,
    log_dir         = LOG_DIR,
)

print('Config:')
for k, v in CFG.items():
    print(f'  {k:<20}: {v}')


## 3 — Imports & reproducibility


In [ ]:
import json
import time
import random
import importlib.util
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from Degrador.PairImages import PairImages

_model_dir = CLEARVISION_PKG / "model"

def _load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_vqvae = _load_module("vqvae_model", _model_dir / "VQ-VAE.py")
_quant = _load_module("quantizer_model", _model_dir / "Quantizer.py")
_loss_mod = _load_module("loss_model", _model_dir / "loss.py")

UNetRestoration = _vqvae.UNetRestoration
VectorQuantizerEMA = _quant.VectorQuantizerEMA
ClearVisionLoss = _loss_mod.ClearVisionLoss


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG["seed"])
print("Imports OK. Seeds set.")


## 4 — Dataset & DataLoaders

In [ ]:
dataset = PairImages(CFG['clean_dir'], CFG['corrupt_dir'])
print(f'Total pairs : {len(dataset)}')

n_val   = max(1, int(len(dataset) * CFG['val_fraction']))
n_train = len(dataset) - n_val

train_ds, val_ds = random_split(
    dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(CFG['seed']),
)
print(f'Train : {n_train}   Val : {n_val}')

train_loader = DataLoader(
    train_ds,
    batch_size  = CFG['batch_size'],
    shuffle     = True,
    num_workers = CFG['num_workers'],
    pin_memory  = (device.type == 'cuda'),
)
val_loader = DataLoader(
    val_ds,
    batch_size  = CFG['batch_size'],
    shuffle     = False,
    num_workers = CFG['num_workers'],
    pin_memory  = (device.type == 'cuda'),
)
print('DataLoaders ready.')


## 5 — Sanity-check: visualise a batch

In [ ]:
corrupted_batch, clean_batch = next(iter(train_loader))
print(f'Corrupted batch : {corrupted_batch.shape}  min={corrupted_batch.min():.3f}  max={corrupted_batch.max():.3f}')
print(f'Clean batch     : {clean_batch.shape}  min={clean_batch.min():.3f}  max={clean_batch.max():.3f}')

def show_pairs(corrupted, clean, n=4):
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
    for i in range(n):
        axes[0, i].imshow(corrupted[i].permute(1, 2, 0).cpu().clamp(0,1))
        axes[0, i].set_title('Corrupted', fontsize=9)
        axes[0, i].axis('off')
        axes[1, i].imshow(clean[i].permute(1, 2, 0).cpu().clamp(0,1))
        axes[1, i].set_title('Clean', fontsize=9)
        axes[1, i].axis('off')
    plt.suptitle('Sample pairs from train set', fontsize=11)
    plt.tight_layout()
    plt.show()

show_pairs(corrupted_batch, clean_batch, n=4)

## 6 — Build model, loss, optimiser

In [ ]:
embedding_dim = CFG['base'] * 16   # 512 for base=32

quantizer = VectorQuantizerEMA(
    num_embeddings  = CFG['num_embeddings'],
    embedding_dim   = embedding_dim,
    commitment_beta = CFG['commitment_beta'],
    decay           = CFG['ema_decay'],
)

model = UNetRestoration(
    in_channels  = 3,
    out_channels = 3,
    base         = CFG['base'],
    quantizer    = quantizer,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters : {total_params:,}')

# ── VRAM estimate ──────────────────────────────────────────────────────────
dummy = torch.randn(1, 3, 128, 128, device=device)
with torch.no_grad():
    _ = model(dummy)
vram_used = torch.cuda.memory_allocated() / 1e9
print(f'VRAM after 1 forward : {vram_used:.2f} GB  (single sample, no AMP)')
del dummy
torch.cuda.empty_cache()

In [ ]:
criterion = ClearVisionLoss(
    lambda_l1   = CFG['lambda_l1'],
    lambda_l2   = CFG['lambda_l2'],
    lambda_perc = CFG['lambda_perc'],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = CFG['lr'],
    weight_decay = CFG['weight_decay'],
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6
)

# AMP scaler — cuts VRAM ~40%, speeds up T4 significantly
scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

print('Model, criterion, optimiser, scaler — all ready.')


## 7 — Metrics helpers
PSNR and SSIM inline (no external dep). LPIPS via the `lpips` package.

In [ ]:
from torchvision.transforms.functional import rgb_to_grayscale

def psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    """Peak Signal-to-Noise Ratio. Both tensors in [0,1]."""
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return 100.0
    return 10 * np.log10(1.0 / mse)


def ssim_map(pred: torch.Tensor, target: torch.Tensor,
             C1=0.01**2, C2=0.03**2) -> torch.Tensor:
    """Per-image SSIM, returned as a (B,) tensor."""
    # Convert to grayscale for standard SSIM
    p = rgb_to_grayscale(pred)    # (B,1,H,W)
    t = rgb_to_grayscale(target)  # (B,1,H,W)

    kernel_size = 11
    sigma = 1.5
    # Gaussian kernel
    coords = torch.arange(kernel_size, dtype=torch.float32, device=pred.device)
    coords -= kernel_size // 2
    g = torch.exp(-(coords**2) / (2 * sigma**2))
    g /= g.sum()
    kernel = (g.unsqueeze(0) * g.unsqueeze(1)).unsqueeze(0).unsqueeze(0)  # (1,1,k,k)

    pad = kernel_size // 2
    mu_p  = nn.functional.conv2d(p, kernel, padding=pad)
    mu_t  = nn.functional.conv2d(t, kernel, padding=pad)
    mu_pp = nn.functional.conv2d(p*p, kernel, padding=pad) - mu_p**2
    mu_tt = nn.functional.conv2d(t*t, kernel, padding=pad) - mu_t**2
    mu_pt = nn.functional.conv2d(p*t, kernel, padding=pad) - mu_p*mu_t

    num = (2*mu_p*mu_t + C1) * (2*mu_pt + C2)
    den = (mu_p**2 + mu_t**2 + C1) * (mu_pp + mu_tt + C2)
    return (num / den).mean(dim=[1,2,3])  # (B,)


# ── LPIPS (optional) ──────────────────────────────────────────────────────────
try:
    import lpips as lpips_lib
    lpips_fn = lpips_lib.LPIPS(net='alex').to(device)
    lpips_fn.eval()
    USE_LPIPS = True
    print('LPIPS : enabled (AlexNet backbone)')
except ImportError:
    USE_LPIPS = False
    print('LPIPS : disabled (pip install lpips to enable)')


@torch.no_grad()
def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> dict:
    """Returns dict with psnr, ssim, (lpips if available). Inputs in [0,1]."""
    out = {}
    out['psnr'] = psnr(pred, target)
    out['ssim'] = ssim_map(pred, target).mean().item()
    if USE_LPIPS:
        # LPIPS expects inputs in [-1, 1]
        p_scaled = pred   * 2 - 1
        t_scaled = target * 2 - 1
        out['lpips'] = lpips_fn(p_scaled, t_scaled).mean().item()
    return out

print('Metric functions defined.')

## 8 — Checkpoint helpers

In [ ]:
def save_checkpoint(path, epoch, model, optimizer, scheduler, scaler, best_psnr, history):
    """
    state_dict() captures parameters AND registered buffers.
    EMA statistics (ema_count, ema_weight, embedding) are buffers
    → saved automatically, no special handling.
    """
    torch.save({
        'epoch'    : epoch,
        'model'    : model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler'   : scaler.state_dict() if scaler is not None else None,
        'best_psnr': best_psnr,
        'cfg'      : CFG,
    }, path)
    with open(os.path.join(CFG['log_dir'], 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)


def load_checkpoint(path, model, optimizer, scheduler, scaler):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'], strict=True)
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    if scaler is not None and ckpt.get('scaler') is not None:
        scaler.load_state_dict(ckpt['scaler'])
    print(f'Resumed from epoch {ckpt["epoch"]}  best PSNR {ckpt["best_psnr"]:.2f}')
    return ckpt['epoch'] + 1, ckpt['best_psnr']

print('Checkpoint helpers defined.')


## 9 — Resume from checkpoint (skip if training fresh)

In [ ]:
RESUME = False   # ← set False to train from scratch

start_epoch = 1
best_psnr   = -1.0
history     = []

last_ckpt = os.path.join(CFG['ckpt_dir'], 'last.pt')

if RESUME and os.path.exists(last_ckpt):
    start_epoch, best_psnr = load_checkpoint(
        last_ckpt, model, optimizer, scheduler, scaler
    )
    hist_path = os.path.join(CFG['log_dir'], 'history.json')
    if os.path.exists(hist_path):
        with open(hist_path) as f:
            history = json.load(f)
    print(f'History loaded: {len(history)} epochs so far')
else:
    print('Starting fresh from epoch 1.')

## 10 — Train & validate functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    sums = dict(loss_total=0., loss_l1=0., loss_l2=0.,
                loss_perc=0., loss_vq=0., perplexity=0.)
    n = 0

    for corrupted, clean in tqdm(loader, desc='train', leave=False):
        corrupted = corrupted.to(device, non_blocking=True)
        clean     = clean.to(device,     non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            recon, vq_loss, perplexity = model(corrupted)
            total_loss, breakdown      = criterion(recon, clean, vq_loss)

        if scaler is not None:
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        bs = corrupted.size(0)
        n += bs
        for k in sums:
            if k == 'perplexity':
                sums[k] += (perplexity.item() if perplexity is not None else 0.) * bs
            elif k in breakdown:
                sums[k] += breakdown[k] * bs

    return {k: v / max(n, 1) for k, v in sums.items()}


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    sums = dict(loss_total=0., perplexity=0., psnr=0., ssim=0., lpips=0.)
    n = 0
    lpips_n = 0

    for corrupted, clean in tqdm(loader, desc='val', leave=False):
        corrupted = corrupted.to(device, non_blocking=True)
        clean     = clean.to(device,     non_blocking=True)

        recon, vq_loss, perplexity = model(corrupted)
        total_loss, _              = criterion(recon, clean, vq_loss)
        m                          = compute_metrics(recon, clean)

        bs = corrupted.size(0)
        n += bs
        sums['loss_total']  += total_loss.item() * bs
        sums['perplexity']  += (perplexity.item() if perplexity is not None else 0.) * bs
        sums['psnr']        += m['psnr'] * bs
        sums['ssim']        += m['ssim'] * bs
        if 'lpips' in m:
            sums['lpips'] += m['lpips'] * bs
            lpips_n       += bs

    out = {k: v / max(n, 1) for k, v in sums.items() if k != 'lpips'}
    if lpips_n:
        out['lpips'] = sums['lpips'] / lpips_n
    return out

print('train_one_epoch and validate defined.')


## 11 — Training loop
Saves checkpoints locally each epoch.


In [ ]:
TARGET_PSNR  = 31.59
TARGET_SSIM  = 0.8415
TARGET_LPIPS = 0.113

print(f'Training from epoch {start_epoch} to {CFG["epochs"]}')
print(f'Target → PSNR ≥ {TARGET_PSNR}  SSIM ≥ {TARGET_SSIM}  LPIPS ≤ {TARGET_LPIPS}')
print('-' * 90)
print(f'{"Ep":>4}  {"T-Loss":>8}  {"V-Loss":>8}  {"PSNR":>7}  '
      f'{"SSIM":>7}  {"LPIPS":>7}  {"Perp":>7}  {"LR":>9}  Flags')
print('-' * 90)

for epoch in range(start_epoch, CFG['epochs'] + 1):
    t0 = time.time()

    train_stats = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
    val_stats   = validate(model, val_loader, criterion)
    scheduler.step()

    elapsed = time.time() - t0
    lr_now  = scheduler.get_last_lr()[0]

    # ── Flags ─────────────────────────────────────────────────────────────
    flags = []
    if val_stats['psnr']  >= TARGET_PSNR:  flags.append('✓PSNR')
    if val_stats['ssim']  >= TARGET_SSIM:  flags.append('✓SSIM')
    lpips_val = val_stats.get('lpips', 999)
    if lpips_val          <= TARGET_LPIPS: flags.append('✓LPIPS')
    perp = val_stats['perplexity']
    if perp < 10:                          flags.append('⚠COLLAPSE')
    flag_str = '  '.join(flags)

    # ── Print row ─────────────────────────────────────────────────────────
    lpips_str = f"{lpips_val:7.4f}" if lpips_val < 999 else '    N/A'
    print(
        f'{epoch:>4}  {train_stats["loss_total"]:>8.4f}  {val_stats["loss_total"]:>8.4f}  '
        f'{val_stats["psnr"]:>7.2f}  {val_stats["ssim"]:>7.4f}  '
        f'{lpips_str}  {perp:>7.1f}  {lr_now:>9.2e}  {flag_str}'
    )

    # ── History ───────────────────────────────────────────────────────────
    history.append({
        'epoch'     : epoch,
        'train'     : train_stats,
        'val'       : val_stats,
        'lr'        : lr_now,
        'elapsed_s' : elapsed,
    })

    # ── Checkpoint every epoch → disk ────────────────────────────────────
    save_checkpoint(
        os.path.join(CFG['ckpt_dir'], 'last.pt'),
        epoch, model, optimizer, scheduler, scaler, best_psnr, history,
    )

    if val_stats['psnr'] > best_psnr:
        best_psnr = val_stats['psnr']
        save_checkpoint(
            os.path.join(CFG['ckpt_dir'], 'best.pt'),
            epoch, model, optimizer, scheduler, scaler, best_psnr, history,
        )
        print(f'       ↑ new best PSNR {best_psnr:.2f} dB — best.pt saved')

print('-' * 90)
print(f'Done. Best PSNR: {best_psnr:.2f} dB')

## 12 — Training curves

In [ ]:
epochs_done = [r['epoch'] for r in history]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Clear_Vision — Training curves', fontsize=13)

def _plot(ax, key, label, target=None, lower_better=False, source='val'):
    vals = [r[source][key] for r in history if key in r[source]]
    ep   = epochs_done[:len(vals)]
    ax.plot(ep, vals, label=source)
    if target is not None:
        ax.axhline(target, color='red', linestyle='--', linewidth=1, label=f'target {target}')
    ax.set_title(label)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# Row 0
_plot(axes[0,0], 'loss_total', 'Total Loss',      source='train')
_plot(axes[0,0], 'loss_total', 'Total Loss',      source='val')
_plot(axes[0,1], 'psnr',       'PSNR (dB)',       target=TARGET_PSNR)
_plot(axes[0,2], 'ssim',       'SSIM',            target=TARGET_SSIM)

# Row 1
_plot(axes[1,0], 'lpips',      'LPIPS ↓',         target=TARGET_LPIPS, lower_better=True)
_plot(axes[1,1], 'perplexity', 'Codebook Perplexity')
axes[1,1].axhline(CFG['num_embeddings'], color='green', linestyle=':', label=f'K={CFG["num_embeddings"]}')
axes[1,1].axhline(10, color='red', linestyle='--', label='collapse threshold')
axes[1,1].legend(fontsize=8)

# Loss breakdown
for key, label in [('loss_l1','L1'), ('loss_l2','L2'),
                   ('loss_perc','Perceptual'), ('loss_vq','VQ')]:
    vals = [r['train'][key] for r in history if key in r['train']]
    axes[1,2].plot(epochs_done[:len(vals)], vals, label=label)
axes[1,2].set_title('Train Loss Breakdown')
axes[1,2].set_xlabel('Epoch')
axes[1,2].legend(fontsize=8)
axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(CFG['log_dir'], 'training_curves.png')
plt.savefig(plot_path, dpi=120)
plt.show()
print(f'Saved → {plot_path}')

## 13 — Visual results: corrupted → reconstructed → clean

In [ ]:
@torch.no_grad()
def show_results(model, loader, n=4):
    model.eval()
    corrupted, clean = next(iter(loader))
    corrupted = corrupted[:n].to(device)
    clean     = clean[:n].to(device)

    recon, _, _ = model(corrupted)

    fig, axes = plt.subplots(3, n, figsize=(3.5*n, 10))
    row_titles = ['Corrupted (input)', 'Reconstructed (model)', 'Clean (target)']

    for i in range(n):
        imgs = [corrupted[i], recon[i], clean[i]]
        for row, (img, title) in enumerate(zip(imgs, row_titles)):
            axes[row, i].imshow(img.cpu().permute(1,2,0).clamp(0,1))
            axes[row, i].axis('off')
            if i == 0:
                axes[row, i].set_ylabel(title, fontsize=10, rotation=90, labelpad=5)

        # Per-image metrics
        p = psnr(recon[i:i+1], clean[i:i+1])
        s = ssim_map(recon[i:i+1], clean[i:i+1]).item()
        axes[2, i].set_title(f'PSNR {p:.2f} | SSIM {s:.4f}', fontsize=8)

    plt.suptitle('Restoration results (val set)', fontsize=12)
    plt.tight_layout()
    vis_path = os.path.join(CFG['log_dir'], 'visual_results.png')
    plt.savefig(vis_path, dpi=120)
    plt.show()
    print(f'Saved → {vis_path}')

show_results(model, val_loader, n=4)

## 14 — Final metric report

In [ ]:
# Load best checkpoint for final eval
best_path = os.path.join(CFG['ckpt_dir'], 'best.pt')
if os.path.exists(best_path):
    load_checkpoint(best_path, model, optimizer, scheduler, scaler)
    print('Loaded best.pt for final evaluation.')

final = validate(model, val_loader, criterion)

print('\n' + '='*50)
print('  FINAL RESULTS (best checkpoint, val set)')
print('='*50)
metrics = [
    ('PSNR  (dB)',  'psnr',       TARGET_PSNR,  False),
    ('SSIM',       'ssim',        TARGET_SSIM,  False),
    ('LPIPS',      'lpips',       TARGET_LPIPS, True ),
    ('Perplexity', 'perplexity',  None,         False),
]
for name, key, target, lower_better in metrics:
    val = final.get(key, None)
    if val is None:
        print(f'  {name:<14}: N/A')
        continue
    if target is None:
        print(f'  {name:<14}: {val:.4f}')
        continue
    hit = (val <= target) if lower_better else (val >= target)
    mark = '✓' if hit else '✗'
    print(f'  {name:<14}: {val:.4f}   (target {target})  {mark}')
print('='*50)

## 15 — If targets not met: tuning playbook

| Symptom | Fix |
|---|---|
| PSNR plateaus early | Lower LR to 5e-5, train 20 more epochs |
| SSIM lagging | Increase `lambda_perc` to 0.2 |
| LPIPS high | Increase `lambda_perc` to 0.2, check VGG is loading |
| Perplexity < 10 (collapse) | Increase `num_embeddings` to 512 or lower `ema_decay` to 0.95 |
| Perplexity == K (all codes equal) | Normal and healthy |
| Loss NaN | Reduce `lr` to 1e-4, check data range is [0,1] |
| OOM on T4 | Reduce `batch_size` to 8 or set `base=16` |